In [ ]:
import os, subprocess, sys
from pathlib import Path

REPO_URL = "https://github.com/AdityaDeodeshmukh/hcsd-project.git"
BRANCH = "feat/mt-p1-latents"

IN_COLAB = "google.colab" in sys.modules or "COLAB_RELEASE_TAG" in os.environ
WORK = Path("/content") if IN_COLAB else Path.cwd().parent

REPO = WORK / "hcsd-project"


def sh(cmd, cwd=None):
    """Run a shell command and stop the notebook if it fails."""
    print("+", cmd)
    subprocess.run(cmd, shell=True, check=True, cwd=cwd)


if IN_COLAB:
    if not REPO.exists():
        sh(f"git clone -q -b {BRANCH} {REPO_URL} {REPO}")
    else:
        sh(f"git fetch -q origin && git checkout -q {BRANCH} && git pull -q", cwd=REPO)
    sh(f"{sys.executable} setup_env.py", cwd=REPO)

os.chdir(REPO)
sh("git log --oneline -1")
print("Working directory:", Path.cwd())

In [ ]:
if IN_COLAB:
    sh(f"{sys.executable} -m pip install -q insightface onnxruntime-gpu gdown")

In [ ]:
SEEDS = [0, 1, 2]
paths = [REPO / "outputs" / f"seed{s:04d}.png" for s in SEEDS]

if not all(p.exists() for p in paths):
    sh(f"{sys.executable} examples/generate.py --seeds {' '.join(map(str, SEEDS))} --outdir outputs")

for p in paths:
    print(p, p.exists())

In [ ]:
import cv2
import numpy as np
from IPython.display import display
from PIL import Image
from insightface.app import FaceAnalysis
from insightface.utils import face_align

app = FaceAnalysis(
    name="buffalo_l",
    allowed_modules=["detection", "recognition"],
    providers=["CUDAExecutionProvider", "CPUExecutionProvider"],
)
app.prepare(ctx_id=0, det_size=(640, 640))


PAD_RATIO = 0.25   # FFHQ faces fill the frame; padding makes them detectable for SCRFD

def detect_and_align(path):
    """Return (aligned 112x112 BGR crop, buffalo_l embedding, detection score) for the single face in `path`."""
    img_bgr = cv2.imread(str(path))  # InsightFace works on BGR images (OpenCV convention)
    assert img_bgr is not None, f"could not read {path}"
    pad = int(img_bgr.shape[0] * PAD_RATIO)
    img_bgr = cv2.copyMakeBorder(img_bgr, pad, pad, pad, pad, cv2.BORDER_CONSTANT, value=(0, 0, 0))
    faces = app.get(img_bgr)
    assert len(faces) == 1, f"{path}: expected 1 face, found {len(faces)}"
    face = faces[0]
    crop_bgr = face_align.norm_crop(img_bgr, landmark=face.kps, image_size=112)  # 5-point similarity transform
    return crop_bgr, face.normed_embedding, float(face.det_score)


crops, emb_buffalo = [], []
for p in paths:
    crop, emb, score = detect_and_align(p)
    crops.append(crop)
    emb_buffalo.append(emb)
    print(f"{p.name}: det_score={score:.3f}, crop={crop.shape}")
emb_buffalo = np.stack(emb_buffalo)

# The crops should be upright, with the eyes level and at the same position in every crop.
strip = cv2.cvtColor(np.hstack(crops), cv2.COLOR_BGR2RGB)
display(Image.fromarray(strip).resize((strip.shape[1] * 2, strip.shape[0] * 2), Image.NEAREST))

In [ ]:
EF_REPO = WORK / "ElasticFace"
EF_DRIVE_FOLDER = "https://drive.google.com/drive/folders/1q3ws_BQLmgXyiy2msvHummXq4pRqc1rx"
ELASTICFACE_WEIGHTS = WORK / "elasticface_arc" / "295672backbone.pth"
# Shared Drive example:
# from google.colab import drive; drive.mount("/content/drive")
# ELASTICFACE_WEIGHTS = Path("/content/drive/MyDrive/<team folder>/elasticface_arc/295672backbone.pth")

if not EF_REPO.exists():
    sh(f"git clone -q https://github.com/fdbtrs/ElasticFace.git {EF_REPO}")
if not ELASTICFACE_WEIGHTS.exists():
    sh(f"gdown --folder {EF_DRIVE_FOLDER} -O {ELASTICFACE_WEIGHTS.parent}")

print(ELASTICFACE_WEIGHTS, f"{ELASTICFACE_WEIGHTS.stat().st_size / 1e6:.0f} MB")

In [ ]:
import torch

sys.path.insert(0, str(EF_REPO))  # iresnet.py imports utils.countFLOPS from the ElasticFace repo root
from backbones.iresnet import iresnet100

device = "cuda" if torch.cuda.is_available() else "cpu"
ef_model = iresnet100(num_features=512).to(device).eval()
ef_model.load_state_dict(torch.load(ELASTICFACE_WEIGHTS, map_location=device, weights_only=True))


@torch.no_grad()
def elasticface_embed(crops_bgr):
    """Aligned 112x112 BGR crops -> L2-normalised 512-d ElasticFace-Arc embeddings, shape (N, 512)."""
    x = np.stack([cv2.cvtColor(c, cv2.COLOR_BGR2RGB) for c in crops_bgr]).astype(np.float32)
    x = torch.from_numpy(x).permute(0, 3, 1, 2).to(device)
    x = (x / 255.0 - 0.5) / 0.5
    return torch.nn.functional.normalize(ef_model(x), dim=1).cpu().numpy()


emb_ef = elasticface_embed(crops)
print("device:", device, "| embeddings:", emb_ef.shape)

In [ ]:
np.set_printoptions(precision=3, suppress=True)


def cosine_matrix(E):
    return E @ E.T  # rows are L2-normalised, so the dot product is the cosine similarity


def off_diagonal(M):
    return M[~np.eye(len(M), dtype=bool)]


flipped = [cv2.flip(c, 1) for c in crops]
results = {
    "ElasticFace-Arc": (emb_ef, elasticface_embed(flipped)),
    "buffalo_l (WebFace600K)": (emb_buffalo, None),
}

for name, (E, E_flip) in results.items():
    M = cosine_matrix(E)
    print(f"\n{name}\n{M}")
    print("  self-similarity ~1:     ", np.allclose(np.diag(M), 1, atol=1e-3))
    print("  max between seeds:      ", f"{off_diagonal(M).max():.3f}  (expect < ~0.3)")
    if E_flip is not None:
        flip_sim = (E * E_flip).sum(1)
        print("  original vs flipped:    ", flip_sim, " (expect > ~0.8)")